# Допуск: LLMs Get Lost In Multi-Turn Conversation

**Статья:** P. Laban, H. Hayashi, Y. Zhou, J. Neville, 2025 — <https://arxiv.org/abs/2505.06120>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / приложение / цитата) нарушено и как должно быть. Код писать не нужно —
достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются
чтением кода и статьи.

Общие импорты:

In [ ]:
import torch

Инструкция — это исходное полностью определённое задание (`full`, например задача GSM8K) и
его шардированная версия (`shards`: список строк, `shards[0]` — общий замысел, остальные —
уточнения, §3.1). Оценка одной симуляции — число от 0 до 100 (§4.1). Функции, которые
используются, но не показаны, считайте корректными.

## Задание 1. Пять типов симуляции

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Каждая функция возвращает список реплик пользователя: между репликами отвечает ассистент.
Симулятор пользователя для простоты опущен — шарды подаются как есть. Определите, какая
функция строит симуляцию FULL, CONCAT, SHARDED, RECAP и SNOWBALL (§3.3, §7.1, прил. M).
Ошибок в коде нет.

In [ ]:
def conv_a(full, shards):
    return [full]

def conv_b(full, shards):
    return list(shards)

def conv_c(full, shards):
    bullets = "\n".join(f"- {s}" for s in shards)
    return ["Complete the task, taking into account all of the following:\n" + bullets]

def conv_d(full, shards):
    turns = [shards[0]]
    for i in range(1, len(shards)):
        past = "\n".join(f"- {s}" for s in shards[:i])
        turns.append(f"Just to reiterate:\n{past}\n\nAlso,\n{shards[i]}")
    return turns

def conv_e(full, shards):
    bullets = "\n".join(f"- {s}" for s in shards)
    return list(shards) + ["To recap, here is everything I need:\n" + bullets]

**Вопросы.**
1. Зачем нужна функция, которая подаёт в одной реплике те же шарды, что и многоходовая
   версия, если уже есть симуляция с исходной инструкцией? Что авторы заключают из того,
   что в среднем она даёт 95.1 % от результата FULL, и как объясняют, что у слабых моделей
   падение здесь заметнее (§6.1)?
2. Две функции добавляют к многоходовому диалогу повторение уже сказанного. Какая из них
   работает лучше по табл. 2, почему авторы всё же называют её нереалистичной и какой
   вывод о «агентных фреймворках поверх LLM» они делают (§7.1)?
3. Для перевода (§7.3, табл. 4) шардированный диалог, построенный как в `conv_b` (каждый
   ход — ещё два предложения, «переведи всё, что есть»), не хуже однократного. Почему? Какие
   три свойства задачи, по мнению авторов, нужны, чтобы модель «терялась» в разговоре?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Шардированная симуляция

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Одна SHARDED-симуляция (рис. 3). `user_sim` раскрывает не больше одного шарда за ход и
возвращает реплику и номер раскрытого шарда (или `None`), `classify` относит ответ
ассистента к одной из семи стратегий (прил. G), `extract` достаёт ответ из текста, `evaluate`
выставляет оценку 0–100. `task.system` — минимальный контекст задачи (схема БД, описание
API и т. п.). Найдите одно отклонение от статьи.

In [ ]:
def simulate_sharded(task, assistant, user_sim, classify, extract, evaluate):
    system = (task.system + "\nThe user may reveal the task gradually over several turns: "
              "ask clarifying questions and do not answer until the request is complete.")
    messages = [{"role": "system", "content": system}]
    revealed, score = set(), 0.0
    while len(revealed) < len(task.shards):
        utterance, shard_id = user_sim(task, messages, revealed)    # at most one new shard
        if shard_id is not None:
            revealed.add(shard_id)
        messages.append({"role": "user", "content": utterance})
        reply = assistant(messages, temperature=1.0)
        messages.append({"role": "assistant", "content": reply})
        if classify(task, messages) == "answer_attempt":
            score = evaluate(task, extract(task, messages))
            if score == 100:
                break                                # solved: the conversation ends
    return score, messages

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Способности и надёжность

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`scores[i, j]` — оценка j-й симуляции i-й инструкции (по 10 симуляций на инструкцию).
Функция считает средний результат P, способность (aptitude) A и ненадёжность
(unreliability) U. Найдите одно отклонение от §4.2.

In [ ]:
def lic_metrics(scores):
    # scores: (N_instructions, N_runs) in [0, 100], N_runs = 10
    P = scores.mean().item()
    pooled = scores.reshape(-1)                                  # all runs of all instructions
    A = torch.quantile(pooled, 0.9).item()                       # aptitude: best-case runs
    U = (torch.quantile(pooled, 0.9) - torch.quantile(pooled, 0.1)).item()   # unreliability
    return P, A, U

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Температура

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Эксперимент §7.2: меняется температура ассистента (AT) и симулятора пользователя (UT).
`simulate(x, setting, assistant_T, user_T)` возвращает оценку одной симуляции,
`unreliability(scores)` считает U так, как в статье. Найдите один неверный комментарий.

In [ ]:
def temperature_experiment(simulate, instructions, temps=(1.0, 0.5, 0.0), n_runs=20):
    results = {}
    for setting in ("full", "concat", "sharded"):
        user_temps = temps if setting == "sharded" else (None,)   # only SHARDED has a simulated user
        for at in temps:
            for ut in user_temps:
                scores = torch.tensor([[simulate(x, setting, assistant_T=at, user_T=ut)
                                        for _ in range(n_runs)] for x in instructions])
                results[setting, at, ut] = unreliability(scores)
    # With AT = UT = 0 decoding is greedy, so every run replays the same conversation token
    # for token: SHARDED unreliability drops to the single-turn level, so lowering the
    # temperature is enough to make multi-turn conversations reliable
    return results

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Когда модель отвечает и сколько говорит

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Анализ логов SHARDED-симуляций из прил. F. У симуляции `c`: `c.labels` — стратегия каждого
ответа ассистента, `c.n_shards` — число шардов, `c.score` — оценка, `c.n_chars` — суммарная
длина ответов ассистента, `c.n_turns` — число ходов.

In [ ]:
BINS = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

def by_first_attempt(convs):
    out = {b: [] for b in BINS}
    for c in convs:
        first = c.labels.index("answer_attempt")
        progress = (first + 1) / c.n_shards                  # share of the instruction revealed
        out[BINS[min(int(progress * 5 - 1e-9), 4)]].append(c.score)
    return {b: sum(s) / len(s) for b, s in out.items() if s}

def by_verbosity(runs):
    # runs: the 10 simulations of ONE (model, instruction) pair -> 2 per quintile
    ranked = sorted(runs, key=lambda c: c.n_chars / c.n_turns)
    names = ["shortest", "short", "median", "long", "longest"]
    return {q: [c.score for c in ranked[2 * i: 2 * i + 2]] for i, q in enumerate(names)}

**Вопросы.**
1. Почему табл. 6 (по `by_first_attempt`) построена только для Code и Math, хотя задач
   шесть? Для каких задач первый ответ всегда приходится на первый ход и почему?
2. Что показывает табл. 6 (сравните первую и последнюю корзины) и как авторы объясняют
   этот эффект? Можно ли по этой таблице утверждать, что ранний ответ *вызывает* провал?
   Назовите хотя бы одно альтернативное объяснение и скажите, какой из возможных
   конфаундеров таблица контролирует.
3. Зачем в `by_verbosity` длину сравнивают внутри 10 симуляций одной пары (модель,
   инструкция), а не по абсолютной длине по всем логам? На какой задаче в табл. 7 эффект
   обратный и почему, по-вашему?

**Ваш ответ:**

_Напишите ответ здесь._